# Book text classification (text-based)
### Group Members: 
#### Kornel Zaleski X00222004 
#### Lukasz Kiraga X00227628 
#### Blessing Mayindu X00224444
#### Goal: The goal for this project is to predict the book genre based on its summary.

In [1]:
# !pip install numpy
# !pip install matplotlib
# !pip install scikit-learn
# !pip install pandas 


In [2]:
# Importing libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
import joblib
import pandas as pd 
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
import os


## Data Cleaning
### Each dataset will be cleaned before merging

In [26]:
# Load and save the cleaned datasets

# BooksDataset.csv - df1
df1 = pd.read_csv("BooksDataset.csv")
df1 = df1.dropna(subset=['Title', 'Description', 'Category'], how='any')
df1 = df1.drop_duplicates()
df1.to_csv('BooksDatasetClean.csv', encoding="utf-8", index=False)

# goodreads_data.csv - df2
df2 = pd.read_csv("goodreads_data.csv")
df2 = df2.dropna(subset=['Title', 'Description', 'Genres'], how='any')
df2 = df2.drop_duplicates()
df2.to_csv('GoodreadsDatasetClean.csv', encoding="utf-8", index=False)

# google_books_dataset.csv - df3
df3 = pd.read_csv("google_books_dataset.csv")
df3 = df3.dropna(subset=['Title', 'Description', 'Category'], how='any')
df3 = df3.drop_duplicates()
df3.to_csv('GoogleBooksDatasetClean.csv', encoding="utf-8", index=False)

# Display the first few rows of each cleaned dataset to ensure the cleaning process
print(df1.head())
print(df2.head())
print(df3.head())

                                                Title  \
7                          Journey Through Heartsongs   
8                        In Search of Melancholy Baby   
10       The Dieter's Guide to Weight Loss During Sex   
11  Germs : Biological Weapons and America's Secre...   
13  The Good Book: Reading the Bible with Mind and...   

                                               Author  \
7                              Stepanek, Mattie J. T.   
8   Aksyonov, Vassily, Heim, Michael Henry, and Bo...   
10                                     Smith, Richard   
11  Miller, Judith, Engelberg, Stephen, and Broad,...   
13                                    Gomes, Peter J.   

                                          Description  \
7   Collects poems written the eleven-year-old mus...   
8   The Russian author offers an affectionate chro...   
10  A humor classic, this tongue-in-cheek diet pla...   
11  Deadly germs sprayed in shopping malls, bomb-l...   
13  "The Bible and the social

## Data merging


In [27]:

# Merging two csv files

common_columns = df1.columns.intersection(df2.columns).intersection(df3.columns)

final_df = pd.concat(
    [df1[common_columns], df2[common_columns], df3[common_columns]],
    ignore_index=True
)

print(final_df.head())
# Here I am showing that the columns have difference headers . To fix this I need to make them the same and then put them together.
print("DF1 columns:")
print(df1.columns.tolist())

print("\nDF2 columns:")
print(df2.columns.tolist())

print("\nDF3 columns:")
print(df3.columns.tolist())

# Making all the columns lowercase
for df in [df1, df2, df3]:
    df.columns = df.columns.str.lower().str.strip()

common_columns = df1.columns.intersection(df2.columns).intersection(df3.columns)
print(common_columns)

                                               Title  \
0                         Journey Through Heartsongs   
1                       In Search of Melancholy Baby   
2       The Dieter's Guide to Weight Loss During Sex   
3  Germs : Biological Weapons and America's Secre...   
4  The Good Book: Reading the Bible with Mind and...   

                                              Author  \
0                             Stepanek, Mattie J. T.   
1  Aksyonov, Vassily, Heim, Michael Henry, and Bo...   
2                                     Smith, Richard   
3  Miller, Judith, Engelberg, Stephen, and Broad,...   
4                                    Gomes, Peter J.   

                                         Description  
0  Collects poems written the eleven-year-old mus...  
1  The Russian author offers an affectionate chro...  
2  A humor classic, this tongue-in-cheek diet pla...  
3  Deadly germs sprayed in shopping malls, bomb-l...  
4  "The Bible and the social and moral consequenc..

In [30]:
#Here I am matching header names to the df3 
#pandas.DataFrame.rename

df1 = pd.read_csv("BooksDatasetClean.csv")
df2 = pd.read_csv("GoodreadsDatasetClean.csv")
df3 = pd.read_csv("GoogleBooksDatasetClean.csv")
dfs = []
for df_loop in [df1, df2, df3]:
    df_loop.columns = df_loop.columns.str.lower().str.strip()
    dfs.append(df_loop)

df1,df2,df3 = dfs

df2 = df2.rename(columns={
    "book": "title",
    "author": "authors",
    "genres": "categories"
})

df1 = df1.rename(columns={
    "category": "categories",
    "author": "authors"
})
#df1 = df1.rename(columns={"categories": "category"})
#df2 = df2.rename(columns={"categories": "category"})
#df3 = df3.rename(columns={"categories": "category"})

common_columns = df1.columns.intersection(df2.columns).intersection(df3.columns)

print("\nCommon columns:", common_columns)
# Merge
df = pd.concat([df1, df2, df3], ignore_index=True)

print("\nFinal merged dataset:")
print(df.head())
print(df.shape)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df["category"].value_counts().to_string())



Common columns: Index(['title', 'description'], dtype='str')

Final merged dataset:
                                               title  \
0                         Journey Through Heartsongs   
1                       In Search of Melancholy Baby   
2       The Dieter's Guide to Weight Loss During Sex   
3  Germs : Biological Weapons and America's Secre...   
4  The Good Book: Reading the Bible with Mind and...   

                                             authors  \
0                             Stepanek, Mattie J. T.   
1  Aksyonov, Vassily, Heim, Michael Henry, and Bo...   
2                                     Smith, Richard   
3  Miller, Judith, Engelberg, Stephen, and Broad,...   
4                                    Gomes, Peter J.   

                                         description  \
0  Collects poems written the eleven-year-old mus...   
1  The Russian author offers an affectionate chro...   
2  A humor classic, this tongue-in-cheek diet pla...   
3  Deadly germs s

## Feature Selection

In [11]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(df)
print('\nidf values:') 
for ele1, ele2 in zip(tfidf.get_feature_names_out(),
tfidf.idf_): print(ele1, ':', ele2) 
print('\nWord indexes:') 
print(tfidf.vocabulary_)
print('\ntf-idf value:') 
print(result) 
print('\ntf-idf values in matrix form:') 
print(result.toarray())


idf values:
author : 2.7047480922384253
avg : 2.7047480922384253
category : 2.2992829841302607
description : 2.7047480922384253
genres : 2.7047480922384253
num : 2.7047480922384253
publisher : 2.7047480922384253
rating : 2.7047480922384253
ratings : 2.7047480922384253
search : 2.7047480922384253
subtitle : 2.7047480922384253
title : 2.7047480922384253

Word indexes:
{'title': 11, 'author': 0, 'description': 3, 'category': 2, 'publisher': 6, 'genres': 4, 'avg': 1, 'rating': 7, 'num': 5, 'ratings': 8, 'subtitle': 10, 'search': 9}

tf-idf value:
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 13 stored elements and shape (10, 12)>
  Coords	Values
  (0, 11)	1.0
  (1, 0)	1.0
  (2, 3)	1.0
  (3, 2)	1.0
  (4, 6)	1.0
  (5, 4)	1.0
  (6, 1)	0.7071067811865475
  (6, 7)	0.7071067811865475
  (7, 5)	0.7071067811865475
  (7, 8)	0.7071067811865475
  (8, 10)	1.0
  (9, 2)	0.6476888299953735
  (9, 9)	0.761904967498719

tf-idf values in matrix form:
[[0.         0.         0.         0.     